In [1]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.evaluate_models import eval_reg

print("Imports successful")

Imports successful


In [2]:
required_objects = [
    "Xtr_f",
    "Xte_f",
    "ytr",
    "yte",
]

for obj in required_objects:
    print(obj, "exists" if obj in globals() else "MISSING")

Xtr_f MISSING
Xte_f MISSING
ytr MISSING
yte MISSING


In [3]:
for obj in ["X_full", "X_early", "y"]:
    print(obj, "exists" if obj in globals() else "MISSING")

X_full MISSING
X_early MISSING
y MISSING


In [4]:
import pandas as pd

df = pd.read_csv("../data/student-mat.csv")

print(df.shape)
df.head()

(395, 1)


,school;sex;age;address;famsize;Pstatus;Medu;Fedu;Mjob;Fjob;reason;guardian;traveltime;studytime;failures;schoolsup;famsup;paid;activities;nursery;higher;internet;romantic;famrel;freetime;goout;Dalc;Walc;health;absences;G1;G2;G3
0,"GP;""F"";18;""U"";""GT3"";""A"";4;4;""at_home"";""teacher..."
1,"GP;""F"";17;""U"";""GT3"";""T"";1;1;""at_home"";""other"";..."
2,"GP;""F"";15;""U"";""LE3"";""T"";1;1;""at_home"";""other"";..."
3,"GP;""F"";15;""U"";""GT3"";""T"";4;2;""health"";""services..."
4,"GP;""F"";16;""U"";""GT3"";""T"";3;3;""other"";""other"";""h..."


In [5]:
print(df.dtypes)

school;sex;age;address;famsize;Pstatus;Medu;Fedu;Mjob;Fjob;reason;guardian;traveltime;studytime;failures;schoolsup;famsup;paid;activities;nursery;higher;internet;romantic;famrel;freetime;goout;Dalc;Walc;health;absences;G1;G2;G3    object
dtype: object


In [6]:
df = pd.read_csv(
    "../data/student-mat.csv",
    sep=";"
)

In [7]:
print(df.shape)
print(df.columns.tolist())

(395, 33)
['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']


In [8]:
df_encoded = pd.get_dummies(
    df,
    drop_first=True
)

print(df_encoded.shape)

(395, 42)


In [9]:
X_full = df_encoded.drop(columns=["G3"]).copy()

y = df_encoded["G3"].copy()

print("Full-information features:", X_full.shape[1])

Full-information features: 41


In [10]:
leak_cols = [
    c for c in X_full.columns
    if c in ("G1", "G2")
]

X_early = X_full.drop(columns=leak_cols)

print("Early-warning features:", X_early.shape[1])

Early-warning features: 39


In [11]:
from sklearn.model_selection import train_test_split

Xtr_f, Xte_f, ytr, yte = train_test_split(
    X_full,
    y,
    test_size=0.20,
    random_state=42
)

Xtr_e, Xte_e, _, _ = train_test_split(
    X_early,
    y,
    test_size=0.20,
    random_state=42
)

print("Train/Test split complete.")

Train/Test split complete.


In [12]:
print(Xtr_f.shape)
print(Xte_f.shape)
print(ytr.shape)
print(yte.shape)

(316, 41)
(79, 41)
(316,)
(79,)


In [13]:
from sklearn.linear_model import (
    LinearRegression,
    Ridge,
    Lasso,
    ElasticNet,
)

In [14]:
activity_models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(
        alpha=1.0,
        max_iter=10000,
    ),
    "Elastic Net": ElasticNet(
        alpha=1.0,
        l1_ratio=0.5,
        max_iter=10000,
    ),
}

In [16]:
student_name = "Roy Jun"
group_name = "Gssrp 2026"

In [17]:
import pandas as pd

student_results = []

for model_name, model in activity_models.items():

    model.fit(Xtr_f, ytr)

    predictions = model.predict(Xte_f)

    metrics = eval_reg(
        yte,
        predictions
    )

    student_results.append({
        "Student": student_name,
        "Group": group_name,
        "Model": model_name,
        "MAE": metrics["MAE"],
        "RMSE": metrics["RMSE"],
        "R2": metrics["R2"],
    })

    print("=" * 60)
    print("Model:", model_name)
    print(f"MAE: {metrics['MAE']:.4f}")
    print(f"RMSE: {metrics['RMSE']:.4f}")
    print(f"R2: {metrics['R2']:.4f}")

Model: Linear Regression
MAE: 1.6467
RMSE: 2.3784
R2: 0.7241
Model: Ridge
MAE: 1.6354
RMSE: 2.3690
R2: 0.7263
Model: Lasso
MAE: 1.2181
RMSE: 2.0424
R2: 0.7966
Model: Elastic Net
MAE: 1.2555
RMSE: 2.0387
R2: 0.7973


In [18]:
student_results_df = pd.DataFrame(student_results)

student_ranked_results = (
    student_results_df
    .sort_values(
        by="RMSE",
        ascending=True
    )
    .reset_index(drop=True)
)

student_ranked_results.insert(
    3,
    "RMSE Rank",
    range(
        1,
        len(student_ranked_results) + 1
    )
)

student_ranked_results.round(4)

,Student,Group,Model,RMSE Rank,MAE,RMSE,R2
0,Roy Jun,Gssrp 2026,Elastic Net,1,1.2555,2.0387,0.7973
1,Roy Jun,Gssrp 2026,Lasso,2,1.2181,2.0424,0.7966
2,Roy Jun,Gssrp 2026,Ridge,3,1.6354,2.3690,0.7263
3,Roy Jun,Gssrp 2026,Linear Regression,4,1.6467,2.3784,0.7241
